# <font color='#569098' size=20> ProteinMPNN and LigandMPNN on the Cluster
 </font>

Tutorial by Nathaniel Felbinger

Additional Reading:
* [ProteinMPNN Paper](https://pubmed.ncbi.nlm.nih.gov/36108050/),
* [LigandMPNN Paper](https://pubmed.ncbi.nlm.nih.gov/40155723/),
* [Enzyme Design with ProteinMPNN](https://pubmed.ncbi.nlm.nih.gov/38194293/),
* [SolubleMPNN Assessment Paper](https://www.nature.com/articles/s41586-024-07601-y)




---

### **OBJECTIVES**

By the end of this notebook you should be able to:

* <font color='#e59454'>Execute ProteinMPNN and LigandMPNN on your cluster to redesign protein monomers and multimers</font>
* <font color='#e59454'>Adjust model settings and understand how they alter design output</font>
* <font color='#e59454'>Perform basic assessment of sequences produced by ProteinMPNN and LigandMPNN</font>

### **TUTORIAL**
The tutorial will have you perform the following tasks:
* <font color='#e59454'>Perform basic sequence optimization of a protein monomer</font>
* <font color='#e59454'>Optimize the sequence of a multimeric immunogen structure and gain and understanding on how to constrain designs</font>
* <font color='#e59454'>Design the sequence of a protein bound to DNA using LigandMPNN</font>



---

**_Please save a copy of this notebook to your own Google Drive before beginning._**

# <font color='#569098'>What is ProteinMPNN?</font>

ProteinMPNN (and LigandMPNN) are structure-based sequence design tools. They perform a task which some refer to as inverse folding, where the model attempts to create a sequence that best fits a given backbone. Both tools can be used to redesign in both monomer and multimer structure context. LigandMPNN adds the additional capability of being able to take into account certain non-protein atoms when creating a designed sequence.

Both methods were trained on structures from the Protein Data Bank (PDB)

# <font color='#569098'>1. Setup</font>


### Activate into ProteinMPNN environment

Note that conda environments are case-sensitive
To check your installed environments you can run:
conda info --envs


```bash
conda activate proteinmpnn

#conda info --envs


# <font color='#569098'>1. Simple Monomer Sequence Design</font>
Our first task here will be to perform sequence design of the immunoglobulin-binding domain of streptococcal Protein G ([1PGX](https://www.rcsb.org/structure/1PGX)). Well, actually it will be a relaxed AlphaFold2 prediction of the same protein.

Let's get situated a bit before we begin:

*   Navigate to the ProteinMPNN install that you have on your cluster

>*cd /path/to/ProteinMPNN*

*   Create directories where your input pdbs and output fasta files will go:

> *mkdir bootcamp_inputs*

> *mkdir bootcamp_outputs*



*   Get the 1PGX_AF2.pdb model located in the ProteinMPNN_Activity folder to the input pdbs directory of your cluster. You can just download the file, open your terminal/terminal emulator and copy it there with:

> *scp 1PGX_AF2.pdb username@cluster.address:/path/to/ProteinMPNN/bootcamp_inputs*







##1. First Run: Starting with the Basics

We will create a shell script that will allow you to submit ProteinMPNN jobs to your cluster queue. For a small monomeric protein like this, generating sequences with CPU should be completely fine. For those bold enough to test their IT staff, you might be able to even get away with just executing it directly from the command line. **This is not recommended**.

We will first execute a very basic cpu-only run, where we will take the 1PGX model as our input and generate 10 output sequences to bootcamp_outputs. A lot of these scripts will be ripped from the examples folder in the github repo ([here](https://github.com/dauparas/ProteinMPNN/tree/main/examples))

Create the file sub_monomer_1.sh and add the contents below. Edit the slurm header files to match whatever format is required of your cluster. The default file paths should work if this script is in the base ProteinMPNN directory. If you run into pathing issues please ask an instructor or TA for help.  


```bash
#!/bin/bash
#SBATCH --mem=16g
#SBATCH -c 2
#SBATCH --output=example_1.out

#source activate proteinmpnn

path_to_PDB="bootcamp_inputs/1PGX_AF2.pdb"
output_dir="bootcamp_outputs/monomer_run_1"
model_dir="vanilla_model_weights"
model_name="v_48_020"

if [ ! -d $output_dir ]
then
    mkdir -p $output_dir
fi

chains_to_design="A"

python ./protein_mpnn_run.py \
        --pdb_path $path_to_PDB \
        --out_folder $output_dir \
        --num_seq_per_target 10 \
        --sampling_temp "0.2" \
        --seed 68 \
        --path_to_model_weights $model_dir \
        --model_name $model_name \

The run should complete very quickly (< 1 minute). If ProteinMPNN was run succcessfully, there should be an output fasta file in the bootcamp_outputs/monomer_run_1 path. Copy and paste the contents of that output file below

### **Paste the monomer_run_1 output fasta here:**



##2. Second Run: Turning Up the Heat

Congrats on your first successful ProteinMPNN run! Now we will adjust some flags to see how turning the knobs and dials of the model changes its predictions

Here we will adjust the **--sampling_temp** flag. This changes how the model samples the probability distribution of the amino acids at each position. Most published designs will use temperatures in the range of 0.1-0.3, but we will get a little crazy here.

It is common to want to exclude certain amino acids from your output designs. In ProteinMPNN, this can be accomplished by defining an unspaced-list of amino acids with the **--omit_AAs** flag.

For your next run, create a new shell script called **sub_monomer_2.sh**. Add the code below to this shell script and submit it to your cluster queue. Once again, please adjust the slurm headers as needed.

``` bash
#!/bin/bash
#SBATCH --mem=16g
#SBATCH -c 2
#SBATCH --output=example_1.out

#source activate proteinmpnn

path_to_PDB="bootcamp_inputs/1PGX_AF2.pdb"
output_dir="bootcamp_outputs/monomer_run_2"
model_dir="vanilla_model_weights"
model_name="v_48_020"

if [ ! -d $output_dir ]
then
    mkdir -p $output_dir
fi

chains_to_design="A"

python ./protein_mpnn_run.py \
        --pdb_path $path_to_PDB \
        --out_folder $output_dir \
        --num_seq_per_target 10 \
        --sampling_temp "0.8" \
        --omit_AAs "CHX" \
        --seed 68 \
        --path_to_model_weights $model_dir \
        --model_name $model_name \

### **Paste the monomer_run_2 output fasta here:**

##3. Third Run: Switching Things (Models) Up

An adjusted design goal that one might use ProteinMPNN for is to design a more soluble sequence that retains a desired input fold. One might use the SolubleMPNN model weights to accomplish this goal. SolubleMPNN is not a completely different program, one can use it easily by adjusting the **--path_to_model_weights** and **--model_name** flags in ProteinMPNN.  

Create a sub_monomer_3.sh script and copy the contents below into it. Submit the script to your cluster queue, once again making adjustments as needed.

```bash
#!/bin/bash
#SBATCH --mem=16g
#SBATCH -c 2
#SBATCH --output=example_1.out

#source activate proteinmpnn

path_to_PDB="bootcamp_inputs/1PGX_AF2.pdb"
output_dir="bootcamp_outputs/monomer_run_3"
model_dir="soluble_model_weights"
model_name="v_48_020"

if [ ! -d $output_dir ]
then
    mkdir -p $output_dir
fi

chains_to_design="A"

python ./protein_mpnn_run.py \
        --pdb_path $path_to_PDB \
        --out_folder $output_dir \
        --num_seq_per_target 10 \
        --sampling_temp "0.8" \
        --omit_AAs "CHX" \
        --seed 68 \
        --path_to_model_weights $model_dir \
        --model_name $model_name \

### **Paste the monomer_run_3 output fasta here:**

## <font color="#B03030">Monomer Analysis Questions</font>

**Please answer the questions below.**

To help with your analysis, an AI-generated helper script called *count_AA.py* is placed in your ProteinMPNN_activity folder. This script takes in the path to one of these file outputs as an argument and outputs the average amino acid percentage observed across the 10 designed sequences. No extra packages are required.

So to get a count of the amino acids in the of fasta file my_seqs.fa:

python count_AA.py my_seqs.fa


**Question 1.** Compare the sequences of the first and second runs. Qualitatively, how did global score and sequence recovery change between the two? Which set of designs do you think is more likely to be successful?

**Answer:** The global score increased while the sequence recovery was reduced. This is likely due to the increase in temperate, not so much the exclusion of cysteine or histidine. I suspect that the lower temperature design is more likely to fold correctly.

**Question 2.** Which amino acids are absent from the sequences of the second run? Why might we want to exclude these amino acids from our designs?

**Answer:** Histidine, Cysteine and the null "residue" are excluded from the sequence. Each is worth exlcuding for its own reason. Histidine is pH dependent. Cysteines can form disulfide bonds which can greatly affect the resulting structure, and the null "residue" is not a real residue!

**Question 3.** Compare the sequences of the first and third run. Which amino acids are incorporated more often in the sequences of the third run compared to those of the first (> 5% increase)? Do you think incorporating these amino acids more frequently would improve solubility?

**Answer:** In general, it increased the number of charged amino acids and decreased the number of hydrophobic amino acids. Incorporating these amino acids would certainly improve solubility; though, they may not have a positive affect on the structure of the protein.

**Question 4.** Is further in-silico assessment required to predict which of the output sequence sets may be the best? Yes or no.

**Answer:** Absolutely more is necessary. One option is to fold the different sequences using a model like Alphafold and then compare these models to the original backbone. The better Alpha fold recovers the original structure, the better the sequence is likely to be.

# <font color='#569098'>2. Multimer Design</font>
ProteinMPNN is also natively capable of performing sequence design on multimeric protein inputs. Here we will perform a couple runs on the pdb file HCV_E1E2_AF2 (a heterodimer composed of the Hepatitis C Virus E1 and E2 envelope proteins) to explore some more complicated design capabilities of ProteinMPNN.

The HCV E1E2 heterodimer is a difficult immunogen to design. Due to mobile regions of E2 and the fact that the E1/E2 interface is held together by weak interactions, the complex demonstrates high structural heterogeneity. The only way to resolve structures (with relatively poor resolution) is through binding of multimer antibodies.

Please download the HCV_E1E2_AF2 pdb file from the ProteinMPNN_Activity folder and place it into your bootcamp_inputs directory. The next scripts you will be using will execute on CPU only. For those that want some practice configuring their SLURM files, please feel free to attempt to utilize GPU resources.

##1. First Run: Redesigning E2

Design of E1E2 primarily focuses on E2 (chain B), as E1 is likely interacting with the host cell membrane and is not an exposed antigen. We will follow this precedent and only focus on designing E2.

To target E2 exclusively for our designs, we will take advantage of some premade helper script that come with ProteinMPNN to define which chains are fixed and which are designable. The **--chain_id_jsonl** flag is what we will be using here, which is defined using **parse_multiple_chains.py** and **assign_fixed_chains.py** scripts in the helper_scripts folder.

An advantage here is that when E2 is being designed, the model can incorporate the sequence identity of the fixed E1 chain.

Create a script called sub_multimer_1.py. Copy the contents below and execute the script.

```bash
#!/bin/bash
#SBATCH --mem=16g
#SBATCH -c 2
#SBATCH --output=example_1.out

#source activate proteinmpnn

path_to_PDB="bootcamp_inputs/HCV_E1E2_AF2.pdb"
output_dir="bootcamp_outputs/multimer_run_1"
model_dir="vanilla_model_weights"
model_name="v_48_020"

if [ ! -d $output_dir ]
then
    mkdir -p $output_dir
fi

chains_to_design="B"

python ./protein_mpnn_run.py \
    --pdb_path "$path_to_PDB" \
    --pdb_path_chains "$chains_to_design" \
    --out_folder "$output_dir" \
    --num_seq_per_target 10 \
    --sampling_temp "0.2" \
    --seed 68 \
    --path_to_model_weights "$model_dir" \
    --model_name "$model_name"

### **Paste the multimer_run_1 output fasta here:**

##2. Second Run: Avoiding Redesign of Epitopes

In the case of redesigning an immunogen like this, you want to be careful about which residues you change. You will want to avoid disturbing the residue identity of epitopes that antibodies recognize and bind to.

In the E1E2 protein, one of the most important epitopes is called AS412 and is comprised of residues 28-40. We will take advantage of the **make_fixed_positions_dict.py** helper script included with ProteinMPNN to avoid altering residues in the epitope range (and a couple others just outside to be safe).   

**Before beginning the next part, navigate (cd) into the bootcamp_inputs directory. Create a new directory called HCV_input. Move the HCV_E1E2_AF2.pdb file into the HCV_input directory**. We do this because certain helper scripts that the ProteinMPNN authors included require you to pass in a directory of pdb files instead of an individual file.

Create a script called sub_multimer_2.py. Copy the contents below and execute the script

```bash
#!/bin/bash
#SBATCH --mem=16g
#SBATCH -c 2
#SBATCH --output=example_1.out

#source activate proteinmpnn

folder_with_pdbs="bootcamp_inputs/HCV_input"
output_dir="bootcamp_outputs/multimer_run_2"
model_dir="vanilla_model_weights"
model_name="v_48_020"

if [ ! -d $output_dir ]
then
    mkdir -p $output_dir
fi

path_for_parsed_chains=$output_dir"/parsed_pdbs.jsonl"
path_for_assigned_chains=$output_dir"/assigned_pdbs.jsonl"
path_for_fixed_positions=$output_dir"/fixed_pdbs.jsonl"

chains_to_design="B"
fixed_positions="25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43"

python ./helper_scripts/parse_multiple_chains.py --input_path=$folder_with_pdbs --output_path=$path_for_parsed_chains

python ./helper_scripts/assign_fixed_chains.py --input_path=$path_for_parsed_chains --output_path=$path_for_assigned_chains --chain_list "$chains_to_design"

python ./helper_scripts/make_fixed_positions_dict.py --input_path=$path_for_parsed_chains --output_path=$path_for_fixed_positions --chain_list "$chains_to_design" --position_list "$fixed_positions"

python ./protein_mpnn_run.py \
    --jsonl_path $path_for_parsed_chains \
    --chain_id_jsonl $path_for_assigned_chains \
    --fixed_positions_jsonl $path_for_fixed_positions \
    --pdb_path "$path_to_PDB" \
    --pdb_path_chains "$chains_to_design" \
    --out_folder "$output_dir" \
    --num_seq_per_target 10 \
    --sampling_temp "0.2" \
    --seed 68 \
    --path_to_model_weights "$model_dir" \
    --model_name "$model_name"

### **Paste the multimer_run_2 output fasta here:**

##3. Third Run: Create Your Own Design Parameters

With some examples of both monomer and multimeric inputs for ProteinMPNN, you are now tasked with creating your own ProteinMPNN designs to improve the use of E1E2 as an immunogen. When thinking about what parameters to set, some questions you can think about are:

1. Do I need to exclude any amino acids from the final design?

2. What positions must be fixed (epitopes, glycan motifs, disulfide bonds, etc.)

  Structure of E1E2 bound to antibodies: [7T6X](https://www.rcsb.org/structure/7T6X)
  
  Another structure of E1E2 bound to antibodies: [8FSJ](https://www.rcsb.org/structure/8FSJ)

3. Is there a particular region of interest to focus on to narrow the number of changes introduced in the design? Perhaps there is a region known to be highly mobile that could be targeted. Or in the case of E1E2, the interface residues could be targeted to more directly stabilize the whole complex.

4. How much variability do you want in your designs?

Feel free to create your own configuration file or modify a given example. Create a new file called multimer_run_3.sh and put your script there.




### **Paste the multimer_run_3 output fasta here:**


## <font color="#B03030">Multimer Analysis Questions</font>

**Please answer the questions below.**

**Question 1.** Compare the sequences of multimer_run_1 and multimer_run_2. Is there a difference between the global scores and sequence recoveries for the two sets of designs?




**Answer:** The global score and sequence recovery are greater for multimer_run_2. Not by a lot, definitely greater.

**Question 2.** Reviewing the sequences of both multimer_run_1 and multimer_run_2 you may observe large stretches of sequence repeats. Why might these occur? Answers can relate to the local geometry these repeats occur in or explanations related to overall structure quality/complexity.

**Answer:** The regions that have repeats are probably regions where the model has no idea what is going on, and is defaulting the mean guess. Apparently, the mean amino acid is alanine. Sometimes glutamate. The more you know!

**Question 3.** What settings did you introduce in your customized input file? Did you have any particular design goals in mind?

**Answer:** I excluded the cysteine and histidine residues, but otherwise kept it unchanged from multimer_run_2. This, in fact, made things worse!

**Question 4.** Is further in-silico assessment required to predict which of the output sequence sets may be the best? Yes or no.

**Answer:** As always, I would want to fold these sequences using alphafold or anotther sequence to structure prediction model. See if any of these are even usable.

# <font color='#569098'>3. LigandMPNN</font>

>When designing in the context of non-protein molecules including nucleotides, glycans, and metal ions one can use LigandMPNN. LigandMPNN is a separate github repository. But it is backwards compatible with ProteinMPNN in that the model can use its weights and essentially ignore the additional all-atom context LigandMPNN enables. In my personal opinion, the input configuration has been cleaned up a lot to the point where it is more convenient to just use the LigandMPNN repository even if you are just running the ProteinMPNN model.

>LigandMPNN also includes basic modeling (with the help of OpenFold) and side chain packing capabilities! For the sake of limiting complexity of the lesson we will not be exploring these, but feel free to check it out on your own.

>In this segment, we will be redesigning the SAP-1 protein bound to DNA ([1BC8](https://www.rcsb.org/structure/1BC8)). This protein is provided as an example by the authors, so nothing needs to be downloaded.

Inside the LigandMPNN repository, create a new folder called bootcamp_outputs. The commands for individual runs are written out for you. Execute each of the runs and answer the questions below

**Run 1: Default Design with ProteinMPNN Weights**

python run.py \
        --seed 68 \
        --pdb_path "./inputs/1BC8.pdb" \
        --out_folder "./bootcamp_outputs/ligandMPNN_run_1" \
        --batch_size 10

**Run 2: Default Design with ligandMPNN Weights**

python run.py \
	      --model_type "ligand_mpnn" \
        --seed 68 \
        --pdb_path "./inputs/1BC8.pdb" \
        --out_folder "./bootcamp_outputs/ligandMPNN_run_2" \
        --batch_size 10

**Run 3: Biasing Sequence Outputs**

python run.py \
	      --model_type "ligand_mpnn" \
        --seed 68 \
        --pdb_path "./inputs/1BC8.pdb" \
        --bias_AA "W:3.0,P:3.0,C:3.0,A:-3.0" \
        --out_folder "./bootcamp_outputs/ligandMPNN_run_3" \
        --batch_size 10

**Run 4: Targeted Residue Design**

python run.py \
	    --model_type "ligand_mpnn" \
        --seed 68 \
        --pdb_path "./inputs/1BC8.pdb" \
        --redesigned_residues "C1 C2 C3 C4 C5 C6 C7 C8 C9 C10" \
        --out_folder "./bootcamp_outputs/ligandMPNN_run_4" \
        --batch_size 10

## <font color="#B03030">DNA Binding Protein Analysis Questions</font>

**Please answer the questions below.**

**Question 1.** Compare the sequences of ligandMPNN_run_1 and ligandMPNN_run_2. Is there a difference between the overall confidence and sequence recovery metrics?



**Answer:** The overall confidence is increased during the second run using the ligandMPNN model, and the ligand confidence is increased even more! However, the sequence recovery is overall lower. Not by a lot, but by some. It would be interesting to see which residues are recovered consistently. They may play an outsized role in the structure.

**Question 2.** Go to the AlphaFold3 webserver ([link text](https://alphafoldserver.com/)) and fold the ninth designed sequence of ligandMPNN_run_1 and ligandMPNN_run_2. The DNA sequence is AACTTCCGGT. How do the confidence metrics compare? What regions of the complex display the largest differences in model confidence? Feel free to fold more of the sequences if you would like, it may help support your answer.

**Answer:** The confidence metrics for both are extremely high. The ligandMPNN model generated sequences are slightly better, but everything is 90%+ confidence almost everywhere except for some residues at the start of the peptide.

**Question 3.** Compare the sequence composition of the ligandMPNN_run_2 and ligandMPNN_run_3 designs using the count_AA.py helper script. Which amino acids display the largest changes in occurrence between the two sets of designs?

**Answer:** Unsurpisingly, the amino acids tryptophan, proline and cysteine have been increased massively. Cysteine is now the most common amino acid which seems suspicious! Alanine meanwhile has disappeared from the structure entirely. Biasing the amino acids works!

**Question 4.** Compare the output stats of ligandMPNN_run_4 and ligandMPNN_run_2. Do they differ significantly? Does limiting the residues that are designed improve sequence recovery?

**Answer:** The sequence recovery is technically lower for the fourth run, but that is because the metric is only counting the part of the sequence that was reconstructed. Obviously, most of the sequence is the exact same for the fourth run because they were copied over. Interestingly, if we limit to the part of the sequence that was reconstructed in the fourth run, I think that they are about comparable. Having the entire ground truth sequence doesn't improve the predictions at all!

**Question 5.** Is further in-silico assessment required to predict which of the output sequence sets may be the best? Yes or no.

**Answer:** I would like to put all ten of the run 2 sequences through alphafold first to select which is best, but in general it seems like a game of inches. None of these sequences seems obviously better.